# 69 Optode co-registration visual check

For one subject, run `ColoredStickerProcessor` with the three-colour
HSV configuration (yellow optodes 'O', green detectors 'G', magenta
sources 'M') on both the original and the anonymized scan, then
render the two side by side in linked PyVista panels.

No matching, no pairing. For each detected sticker on the original
we just check whether the anonymized scan has any detection within
`MISSING_RADIUS_MM`. Originals with no anonymized counterpart in
that radius are coloured **red** so they stand out -- visually
verify whether each red sticker sits on a region the anonymization
deleted (face / ears) or on the cap (would indicate a real problem).


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pyvista as pv
from scipy.spatial import KDTree

import cedalion
import cedalion.vis.blocks as vbx
from cedalion.geometry.landmarks import normalize_landmarks_labels
from cedalion.geometry.photogrammetry.processors import ColoredStickerProcessor

from _thesis_data import load_raw, load_anon, load_landmarks

pv.set_jupyter_backend('server')

SUBJECT = 17

# Same HSV ranges _validator_coreg.py used:
#   O = yellow optode stickers
#   G = green detector stickers
#   M = magenta source stickers
COLORS = {
    'O': (0.11, 0.21, 0.70, 1.0),
    'G': (0.14, 0.30, 0.30, 1.0),
    'M': (0.85, 1.00, 0.40, 1.0),
}

# Per group: an original sticker is treated as 'missing on the anon
# side' if no anon detection of the same colour group lies within
# this radius. Detector reproducibility noise is sub-millimetre, so
# a few mm threshold cleanly separates 'noisy match' from 'no
# counterpart at all'.
MISSING_RADIUS_MM = 3.0

GROUP_COLOR = {'O': '#ffaa00', 'G': '#22cc22', 'M': '#cc22cc'}
MISSING_COLOR = '#ff0000'


## 1. Load surfaces and landmarks

Subject17 is the post-dev-merge verified subject. Change `SUBJECT`
in the previous cell to re-run on any other optode-cohort subject
(16, 18, 19, 20, 21, 22).


In [2]:
surface_orig = load_raw(SUBJECT)
surface_anon = load_anon(SUBJECT)
landmarks = normalize_landmarks_labels(load_landmarks(SUBJECT))

print(f'Subject{SUBJECT}')
print(f'  original  vertices: {len(surface_orig.mesh.vertices):,}')
print(f'  anonymized vertices: {len(surface_anon.mesh.vertices):,}')
print(f'  landmarks: {list(landmarks.label.values)}')


Subject17
  original  vertices: 1,284,667
  anonymized vertices: 593,963
  landmarks: [np.str_('Nz'), np.str_('Iz'), np.str_('Cz'), np.str_('LPA'), np.str_('RPA')]


## 2. Detect stickers on both scans, split by colour group

One `ColoredStickerProcessor.process` call per surface with all
three HSV ranges, then split the returned centres by their `group`
coordinate.


In [3]:
def detect_by_group(surface):
    proc = ColoredStickerProcessor(colors=COLORS)
    centres, _normals = proc.process(surface, details=False)
    if len(centres) == 0:
        return {}
    c_np = centres.pint.dequantify().values
    g_arr = np.asarray(centres.group.values)
    return {str(g): c_np[g_arr == g] for g in np.unique(g_arr)}

groups_orig = detect_by_group(surface_orig)
groups_anon = detect_by_group(surface_anon)

for g in sorted(set(groups_orig) | set(groups_anon)):
    no = len(groups_orig.get(g, np.empty((0, 3))))
    na = len(groups_anon.get(g, np.empty((0, 3))))
    print(f'  group {g}: original={no:3d}, anonymized={na:3d}')


[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [195.392883 145.783371 477.122894]
 ...
 [168.050629   2.322266 444.530457]
 [168.193817   2.722275 444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[172.471527 177.925201 494.870483]
 [186.736481  67.19873  514.295105]
 [140.480835 -19.345001 457.866455]
 ...
 [187.237213  33.922272 444.530457]
 [187.257904  33.522263 444.530457]
 [187.015991  33.060547 444.626129]]
G (0.14, 0.3, 0.3, 1.0)


[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[ 104.91983   218.959656  476.566742]
 [-239.395691  -25.133606  518.167358]
 [-201.389893  -14.077728  528.530457]
 ...
 [-185.138641  -18.929413  583.291687]
 [-244.212967  -44.16188   582.381287]
 [-240.31311   -41.677734  582.530457]]
M (0.85, 1.0, 0.4, 1.0)
[0.02940559 0.95909248 2.11015511 0.7870147  0.90299782 1.0188369
 0.78557518 0.22357185 0.38533485 0.09604622 1.72756279 0.34873634
 1.88083138 0.87307441 0.52633369 0.63046943 1.3329522  1.06282769
 0.47782952 2.15998284 0.06028175 0.34719841 0.55752113 0.26658088
 0.20127052 0.08177361 0.48711632 0.78656883 0.00957512 0.89898262
 0.37162316 1.8688401  0.91721184 0.35123298 0.19180962 0.15842186
 0.15517921 0.07921333 0.24855446 0.37246666 0.95909691 0.55750252
 1.88079354 0.43935707 0.21303124 0.22787727 1

[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [173.665268 152.718643 523.777771]
 ...
 [191.205231  50.322266 444.530457]
 [191.330841  50.72226  444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[172.471527 177.925201 494.870483]
 [186.736481  67.19873  514.295105]
 [170.163727 158.72226  523.730469]
 ...
 [187.237213  33.922272 444.530457]
 [187.257904  33.522263 444.530457]
 [187.015991  33.060547 444.626129]]
G (0.14, 0.3, 0.3, 1.0)


[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[104.91983  218.959656 476.566742]
 [117.292023 200.184784 514.930481]
 [116.492035 200.297455 514.930481]
 ...
 [187.158783  33.927765 459.793518]
 [130.975403 209.783035 430.359772]
 [191.995453 154.842987 446.893677]]
M (0.85, 1.0, 0.4, 1.0)
[0.32077221 1.00696094 2.0571394  0.88362262 1.35326367 1.04817852
 0.23729847 0.33489453 0.20170239 1.14434033 1.23635235 0.41266803
 1.97634503 0.85782795 0.5323161  0.81556507 1.38189709 1.25476056
 0.35434613 2.08681897 0.09497632 0.21938305 0.75328479 0.24899768
 0.40391986 0.12973156 0.22483471 1.82561673 0.5472395  0.14191906
 0.77463825 0.28751934 1.87140308 0.81853802 0.11845935 0.13803662
 0.02745197 0.21372058 0.44797046 0.09909222 0.92190472 0.60756336
 0.14160053 1.89238125 0.6579198  0.25093738 1.74967762 0.10776

## 3. Flag originals with no anon counterpart

Per group, for each original sticker centre, compute the distance
to the nearest anon sticker of the same group (KDTree). If it
exceeds `MISSING_RADIUS_MM`, flag it as missing on the anon side.
No matching, no 1:1 constraint, each original is checked
independently.


In [4]:
def missing_mask(c_orig, c_anon, radius=MISSING_RADIUS_MM):
    if len(c_orig) == 0:
        return np.empty(0, dtype=bool)
    if len(c_anon) == 0:
        return np.ones(len(c_orig), dtype=bool)
    tree = KDTree(c_anon)
    dist, _ = tree.query(c_orig)
    return dist > radius

missing = {}  # group -> bool array over groups_orig[g]
for g in sorted(groups_orig):
    m = missing_mask(groups_orig[g], groups_anon.get(g, np.empty((0, 3))))
    missing[g] = m
    print(f'  group {g}: {int(m.sum())} of {len(m)} original stickers '
          f'have no anon counterpart within {MISSING_RADIUS_MM} mm')


  group G: 1 of 126 original stickers have no anon counterpart within 3.0 mm
  group M: 0 of 2 original stickers have no anon counterpart within 3.0 mm
  group O: 0 of 21 original stickers have no anon counterpart within 3.0 mm


## 4. Side-by-side detections

Two linked PyVista panels. Left: original. Right: anonymized.

- **Yellow / green / magenta spheres** -- detections in their
  natural group colour ('O' / 'G' / 'M').
- **Red spheres** (left panel only) -- original detections with no
  anon counterpart within `MISSING_RADIUS_MM`. Spin the view and
  check where each red sphere sits: face / nose / ears means the
  anonymization correctly deleted the underlying mesh; cap region
  means a real cap sticker disappeared and warrants investigation.
- **Cyan labelled spheres** -- the five 10-20 landmarks for
  orientation.

`link_views()` ties the cameras so rotating one panel rotates the
other.


In [5]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

# Left: original, with missing-on-anon stickers in red.
pvplt.subplot(0, 0)
n_missing = sum(int(m.sum()) for m in missing.values())
pvplt.add_text(f'Original (Subject{SUBJECT}) -- {n_missing} red = no anon match',
               font_size=10)
vbx.plot_surface(pvplt, surface_orig, opacity=1.0)
for g, c in groups_orig.items():
    if len(c) == 0:
        continue
    m = missing[g]
    if (~m).any():
        pvplt.add_points(c[~m], color=GROUP_COLOR.get(g, 'white'),
                         point_size=14, render_points_as_spheres=True)
    if m.any():
        pvplt.add_points(c[m], color=MISSING_COLOR,
                         point_size=18, render_points_as_spheres=True)
vbx.plot_labeled_points(pvplt, landmarks, color='cyan', show_labels=True)

# Right: anonymized, all detections in their group colour.
pvplt.subplot(0, 1)
n_anon = sum(len(c) for c in groups_anon.values())
pvplt.add_text(f'Anonymized (Subject{SUBJECT}) -- {n_anon} stickers',
               font_size=10)
vbx.plot_surface(pvplt, surface_anon, opacity=1.0)
for g, c in groups_anon.items():
    if len(c) == 0:
        continue
    pvplt.add_points(c, color=GROUP_COLOR.get(g, 'white'),
                     point_size=14, render_points_as_spheres=True)
vbx.plot_labeled_points(pvplt, landmarks, color='cyan', show_labels=True)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:39919/index.html?ui=P_0x7f0bd10a5990_0&reconnect=auto" class="pyvi…

## 5. What to look for

Spin the linked view and locate every red sphere on the original
panel:

- **Red on the face / nose / cheeks / ears** -- expected. The
  anonymization deleted those vertices, so any sticker the colour
  detector picked up there cannot have a counterpart on the anon
  scan. Harmless.
- **Red on the cap** -- not expected. A real cap sticker lost its
  detection on the anonymized side. Possible causes worth checking:
  cap-boundary detector clipped too low and removed cap vertices,
  or `ColoredStickerProcessor` clustering happened to fail on a
  near-threshold sticker on one of the two scans.
- **Red just at the cap edge** -- borderline. Likely the
  cap-boundary detector trimmed a few mm of the very lowest cap
  row; cross-check against the cap-boundary discussion in the
  thesis.
